# 04 · Model 1: therapeutic escalation (pipeline test)

Binary outcome: escalation within 12 months of starting topical treatment.
Patients censored before 12 months without escalation are excluded from this binary version (the time-to-event version will use them).

⚠️ For now this runs **only on fictitious data**, to check that the code works. Do not interpret the numbers.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from src.config import params, path
pd.set_option("display.max_columns", 50)

In [ ]:
# Data source: "fictitious" (test data) or "kruczek_il17" (Polish public dataset)
SOURCE = "fictitious"

if SOURCE == "fictitious":
    from src.data.loaders.fictitious import load
elif SOURCE == "kruczek_il17":
    from src.data.loaders.kruczek_il17 import load

from src.data.clean import clean
raw = load()
tables, cleaning_log = clean(raw)
{k: len(v) for k, v in tables.items()}

In [ ]:
from src.data.cohorts import model1_cohort
from src.outcomes.escalation import compute_escalation
from src.features.baseline import model1_features

c1 = model1_cohort(tables)
esc = compute_escalation(tables, c1)
X_all = model1_features(tables, c1)
d = X_all.merge(esc, on="patient_id")
OUTCOME = "escalation_365d"
d = d.dropna(subset=[OUTCOME])
print(f"{len(d)} patients, {int(d[OUTCOME].sum())} escalations")

In [ ]:
NUM = ["age", "bmi", "disease_duration", "pasi_baseline", "bsa_baseline", "dlqi_baseline",
       "sex_male", "difficult_sites", "psoriatic_arthritis", "obesity", "diabetes", "family_history"]
CAT = []
X, y, groups = d[NUM + CAT], d[OUTCOME].astype(int), d.patient_id

from src.models.baseline import null_model, logistic
from src.evaluation.cv import evaluate, summarize
P = params()
res = {}
for name, m in {"null": null_model(), "logistic": logistic(NUM, CAT, seed=P["seed"])}.items():
    met, _ = evaluate(m, X, y, groups, n_splits=P["cv"]["n_splits"], n_repeats=P["cv"]["n_repeats"], seed=P["seed"])
    res[name] = summarize(met)
pd.DataFrame(res).T

Next step for Model 1 (protocol): time-to-event model (e.g. Cox) with C-index, using `event` and `time_days`.